<a href="https://colab.research.google.com/github/AlishbaMalik687-svg/AI-ML_internship_Adv_Task_2/blob/master/Adv_task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Enable GPU

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

 2. Install libraries

In [ ]:

!pip install transformers datasets gradio -q

# 3. Import libraries

from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score




 3. Loading Dataset


In [ ]:

dataset = load_dataset("HuyAugie/Smaller_AG_News_Dataset")

train_dataset = dataset["train"]
test_dataset = dataset["test"]



4. Load Tokenizer

In [ ]:

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")   #Yahan hum bert-base-uncased ka tokenizer load kar rahe hain.Text ko numbers (tokens) me convert karna.


 5. Tokenization Function

In [ ]:
 # Yahan hum dataset ke har news text ko tokenizer se process kar rahe hain.
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,     # truncation=True → agar text lamba ho to cut ho jaye
        padding="max_length",     #padding="max_length" → sab inputs equal length ke ho jayein
        max_length=128
    )


# Apply tokenization
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)




6. Load BERT Model

In [ ]:
#Yahan hum BERT model load kar rahe hain jo already pretrained hai.
model = BertForSequenceClassification.from_pretrained(
   "bert-base-uncased",
    num_labels=4      #Matlab model 4 categories predict karega.
)

 #Hum isko fine-tune kar rahe hain news classification ke liye



 7. Training Arguments

In [ ]:
 #Yahan hum training settings define kar rahe hain.
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,        #model itni fast learn kare
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,      #Matlab model dataset 3 times train karega.
    logging_dir="./logs"
)

In [ ]:
# is function se accuracy or fi scor pta chaly ga model ka

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)

    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')

    return {
        "accuracy": acc,
        "f1_score": f1
    }

 8. Trainer

In [ ]:
#Trainer Hugging Face ka tool hai jo: training handle karta hai, evaluation karta hai, logging karta hai
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)
 #yahan hum: training dataset, testing dataset, model, sab connect kar rahe hain.

9. Train Model

In [ ]:
 #Ab model training start karta hai.
trainer.train()

10.send model to device to avoid GPU and CPU mismatch error

In [ ]:
model.to(device)

11. Evaluate Model

In [ ]:
#yahan model test dataset par check hota hai.
result = trainer.evaluate()
print("Evaluation Result:", result)

12. Gradio Interface

In [ ]:
#Is se simple web app ban jata hai jahan user news headline enter karta hai aur model topic predict karta hai.
import gradio as gr

labels = ["World", "Sports", "Business", "Sci/Tech"]

def predict(text):
    model.eval() # Set model to evaluation mode

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # Move tokenized inputs to the device
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    pred = outputs.logits.argmax().item()

    return labels[pred]


demo = gr.Interface(
    fn=predict,
    inputs="text",
    outputs="text",
    title="News Topic Classifier"
)

demo.queue()
demo.launch(share=True, debug=True)